# 🔧 CTDDG — Cluster Setup & Environment VerificationRun this notebook **first** to verify your Bhavani cluster environment.**Prerequisites:** The `ctddg_env` conda environment must already exist.Run `bash cluster/setup_jupyter_kernel.sh` on the login node first.

In [ ]:
import os, sys, subprocess# ── Auto-detect project root ──CTDDG_ROOT = os.environ.get("CTDDG_ROOT", os.path.dirname(os.getcwd()))os.environ["CTDDG_ROOT"] = CTDDG_ROOTos.chdir(CTDDG_ROOT)sys.path.insert(0, os.path.join(CTDDG_ROOT, "scripts"))print(f"Project Root: {CTDDG_ROOT}")print(f"Python:       {sys.executable}")print(f"Python ver:   {sys.version}")

## 1. GPU Verification

In [ ]:
# Check NVIDIA GPUs!nvidia-smiprint()import mxnet as mxnum_gpus = mx.context.num_gpus()print(f"\n✅ MXNet {mx.__version__} — {num_gpus} GPU(s) detected")# Quick GPU compute testfor i in range(num_gpus):    x = mx.nd.ones((100, 100), ctx=mx.gpu(i))    y = mx.nd.dot(x, x)    print(f"  GPU {i}: {y[0,0].asscalar():.0f} (dot product test passed)")

## 2. Dependency Verification

In [ ]:
deps = {}for name, imp in [("mxnet", "mxnet"), ("numpy", "numpy"), ("scipy", "scipy"),                   ("rdkit", "rdkit"), ("pandas", "pandas"), ("networkx", "networkx"),                   ("h5py", "h5py"), ("matplotlib", "matplotlib"), ("molvs", "molvs")]:    try:        m = __import__(imp)        deps[name] = getattr(m, "__version__", "OK")    except ImportError:        deps[name] = "❌ MISSING"for k, v in deps.items():    status = "✅" if v != "❌ MISSING" else "❌"    print(f"  {status} {k:15s} {v}")

## 3. Data Directory Verification

In [ ]:
from scripts.config import cfgcfg.__init__(CTDDG_ROOT)checks = [    ("atom_types.txt", cfg.ATOM_TYPES_FILE),    ("chembl/chembl.txt", cfg.CHEMBL_PLAIN_FILE),    ("bindingdb/train_dataset", os.path.join(cfg.BINDINGDB_DIR, "train_dataset")),    ("bindingdb/test_dataset", os.path.join(cfg.BINDINGDB_DIR, "test_dataset")),]all_ok = Truefor label, path in checks:    exists = os.path.exists(path)    if not exists: all_ok = False    print(f"  {'✅' if exists else '❌'} {label}: {path}")if all_ok:    print("\n✅ All data files present!")else:    print("\n⚠️  Some data files missing. Run download_data.sh or rsync from local.")

## 4. Patch Notebook PathsRun this to update all hardcoded paths in the original notebooks.

In [ ]:
!python scripts/patch_paths.py {CTDDG_ROOT}

## 5. Convert ChEMBL FormatAppend dummy class labels for the pretraining data loader.

In [ ]:
chembl_src = os.path.join(CTDDG_ROOT, "data", "chembl", "chembl.txt")chembl_dst = os.path.join(CTDDG_ROOT, "data", "chembl", "chembl_final.txt")if os.path.exists(chembl_src) and not os.path.exists(chembl_dst):    !python scripts/convert_chembl_format.py {chembl_src} {chembl_dst}elif os.path.exists(chembl_dst):    print(f"✅ chembl_final.txt already exists ({os.path.getsize(chembl_dst)} bytes)")else:    print("❌ chembl.txt not found — download data first")

## 6. Create Output Directories

In [ ]:
for d in ["outputs/pretrain/logs", "outputs/docking"]:    os.makedirs(os.path.join(CTDDG_ROOT, d), exist_ok=True)    print(f"  ✅ {d}/")print("\n🎉 Setup complete! Proceed to notebook 01_pretraining.ipynb")